# Loan fairness audit: does a mortgage model treat Black applicants the same?

**Veritas AI International High School Hackathon 2026 · AI + Finance · suggested project 4.2.3 (fairness audit of a loan model)**

> This is our submission notebook. It runs the whole audit in order, so every number below is computed here rather than copied from somewhere else.

**Team:** _your names here_

**One-sentence pitch:** We take HMDA 2025 mortgage applications from four states, train a model that predicts approval, and measure how much less often it approves Black applicants than White ones — then test whether removing race, or setting a different cutoff per group, actually closes that gap.

**Who it is for:** a housing counselor at a nonprofit, sitting with an applicant whose loan was declined. They need to know whether the decision came from the applicant's finances or from their race, and our audit says how much of it is race.

**How to run:** Runtime → Run all. Needs `pandas`, `numpy` and `scikit-learn`. It uses `data/hmda_2025_ga_md_mi_az.csv` if that file is present, otherwise it downloads a 224 MB public extract from the CFPB once, which takes about a minute and needs no key.

Everything is scored on applications the model never saw in training: 20% of the sample is held out as a test set and never touched until the scores below are printed.

*This project is a prototype and is not financial advice.*

## 1. Load the data

We use **HMDA 2025** (Home Mortgage Disclosure Act) loan-level data from the CFPB Data Browser API: every application in Georgia, Maryland, Michigan and Arizona for a home purchase or a refinance, marked either originated or denied. HMDA is the data US regulators actually use for fair-lending review, so the setting is real rather than a classroom dataset.

Filings are narrowed to applicants coded White or Black who reported an age, then cut to a fixed random sample of 100,000 so the audit finishes in about a minute. `audit/dataset.py` holds the query, the filters and the sample seed.

Only fields that exist **when the application is filed** are kept. Interest rate, rate spread, points and fees, purchaser type and the lender's recorded denial reasons are all reported *after* the decision, so including them would leak the answer.

In [1]:
import time

import numpy as np
import pandas as pd

from audit import dataset

t0 = time.time()
df = dataset.load()          # cached CSV if present, otherwise downloads once from the CFPB
y = df["target"].to_numpy()

print(f"{len(df):,} applications · {y.mean():.1%} approved · loaded in {time.time() - t0:.0f}s")
df[["race", "age_group", "loan_amount", "loan_to_value_ratio", "income",
    "debt_to_income_ratio", "target"]].head()

100,000 applications · 84.5% approved · loaded in 7s


,race,age_group,loan_amount,loan_to_value_ratio,income,debt_to_income_ratio,target
0,White,25 and over,185000.0,100.000,75.0,50%-60%,1
1,White,25 and over,255000.0,23.969,24.0,>60%,0
2,White,25 and over,245000.0,80.000,69.0,46,1
3,White,25 and over,325000.0,76.000,257.0,49,1
4,Black,25 and over,605000.0,88.540,103.0,>60%,0


## 2. Look at the data first

Before training anything, we look at who is in the sample and how often each group was approved in the real data. This is the benchmark the model has to match or beat — not a fair world, the actual one.

In [2]:
def group_table(column):
    table = (
        df.groupby(column)["target"]
        .agg(applications="size", approved="mean")
        .reset_index()
        .rename(columns={column: "group"})
    )
    table["share of sample"] = (table["applications"] / len(df) * 100).round(1).astype(str) + "%"
    table["approved in data"] = (table["approved"] * 100).round(1).astype(str) + "%"
    return table.drop(columns="approved")


print("Race")
display(group_table("race"))
print("Age")
display(group_table("age_group"))

Race


,group,applications,share of sample,approved in data
0,Black,17772,17.8%,74.3%
1,White,82228,82.2%,86.7%


Age


,group,applications,share of sample,approved in data
0,25 and over,94364,94.4%,84.5%
1,under 25,5636,5.6%,84.8%


In the recorded decisions, roughly **87%** of White applicants were approved against **74%** of Black applicants — a gap of about 12 points. That gap is what we are auditing. It is not proof of discrimination on its own (income, credit history and loan size are not in this file), which is exactly why we compare it with what a model does when it is allowed to see race, and when it is not.

## 3. Split into training, validation and test sets

We hold out 20% for testing and never train on it. We also keep a validation slice, because the model's 0.5 cutoff is meaningless at an 84.5% base rate — it would approve almost everyone and hide any gap. Instead we pick the cutoff that makes the model approve the same share of applications the lenders did, and we pick it on validation, never on test.

In [3]:
rng = np.random.default_rng(0)
order = rng.permutation(len(df))
n_test = int(0.2 * len(df))
n_val = int(0.2 * len(df))
test, val, train = order[:n_test], order[n_test:n_test + n_val], order[n_test + n_val:]

for name, part in [("train", train), ("validation", val), ("test", test)]:
    print(f"{name:<11}{len(part):>7,} applications   {y[part].mean():.1%} approved")

train       60,000 applications   84.4% approved
validation  20,000 applications   84.7% approved
test        20,000 applications   84.7% approved


## 4. Baseline: approve every applicant

The simplest possible rule. It gets the base rate right and denies nobody, so it looks harmless and decides nothing at all. Any model we build has to be better than this on accuracy, and we have to look at what it does to the gap.

In [4]:
def report(name, approved, y_true, trait, groups):
    """Accuracy, the approval rate in each group, and the gap in points."""
    other = dataset.GROUPS[trait]["other"]
    ref = dataset.GROUPS[trait]["ref"]
    rate_other = approved[groups == other].mean() * 100
    rate_ref = approved[groups == ref].mean() * 100
    print(name)
    print(f"  accuracy        {(approved == y_true).mean() * 100:5.1f}%")
    print(f"  {other:<15} {rate_other:5.1f}%    {ref:<13} {rate_ref:5.1f}%    gap {rate_other - rate_ref:+.1f}")


baseline = np.ones(len(test), dtype=bool)
report("Baseline: approve every applicant", baseline, y[test], "race", df["race"].to_numpy()[test])

Baseline: approve every applicant
  accuracy         84.7%
  Black           100.0%    White         100.0%    gap +0.0


84.7% accuracy while approving everyone, including the 15.3% the lenders denied in this test set. This is the same trap as always predicting "not spam": a high score that hides the fact that the tool catches nothing. From here on we report the approval rate for each group next to accuracy.

## 5. Model 1: logistic regression that can see race

Numeric columns are median-imputed and scaled, categorical columns are one-hot encoded, and a logistic regression predicts approval. This model sees race, ethnicity, sex and age — the version of the tool that is clearly illegal to ship, used here as the upper bound on how much the demographics move decisions.

In [5]:
from audit.audit import calibrated_cut, make_model

features = dataset.NUMERIC + dataset.CATEGORICAL
blind_features = [c for c in features if c not in dataset.PROTECTED]
X = df[features]

full = make_model(features).fit(X.iloc[train], y[train])
p_full_val = full.predict_proba(X.iloc[val])[:, 1]
p_full_test = full.predict_proba(X.iloc[test])[:, 1]
cut_full = calibrated_cut(y[val], p_full_val)
approved_full = p_full_test >= cut_full

print(f"cutoff tuned on validation: {cut_full:.3f} → approves {approved_full.mean():.1%} of test applications")
for trait in ("race", "age"):
    report(f"Model 1, {trait}", approved_full, y[test], trait,
           df[dataset.GROUPS[trait]["column"]].to_numpy()[test])

cutoff tuned on validation: 0.753 → approves 84.8% of test applications
Model 1, race
  accuracy         87.9%
  Black            69.4%    White          88.1%    gap -18.7
Model 1, age
  accuracy         87.9%
  under 25         85.0%    25 and over    84.8%    gap +0.2


The model is more accurate than the baseline (87.9% against 84.7%) and it **widens** the racial gap: at the same overall approval rate it approves Black applicants 69.4% of the time against 88.1% for White applicants, a gap of −18.7 on this split. Averaged over 50 splits the page reports −19.7, against −12.2 in the lenders' own decisions.

Accuracy and fairness are not the same measurement, and here they point in opposite directions.

## 6. Where Model 1 goes wrong

Two views of failure. First the confusion matrix, then a concrete pair of applications the model treats differently.

In [6]:
from sklearn.metrics import confusion_matrix

counts = confusion_matrix(y[test], approved_full)
pd.DataFrame(counts, index=["denied in data", "approved in data"],
             columns=["model denies", "model approves"])

,model denies,model approves
denied in data,1845,1225
approved in data,1194,15736


In [7]:
from scipy.spatial.distance import cdist

scored = df.iloc[test].copy()
scored["score"] = p_full_test
scored["model approves"] = approved_full

numbers = ["loan_amount", "loan_to_value_ratio", "income", "property_value"]
complete = scored.dropna(subset=numbers + ["debt_to_income_ratio"])
black_denied = complete[(complete["race"] == "Black") & (~complete["model approves"])]
white_approved = complete[(complete["race"] == "White") & complete["model approves"]]

# The closest Black-denied / White-approved pair on the numbers the model uses,
# holding debt-to-income fixed so the two applications differ on race alone.
mean = complete[numbers].mean()
scale = complete[numbers].std().replace(0, 1)
best = None
keys = set(zip(black_denied["debt_to_income_ratio"], black_denied["age_group"])) & set(
    zip(white_approved["debt_to_income_ratio"], white_approved["age_group"])
)
for dti, age in keys:
    b = black_denied[(black_denied["debt_to_income_ratio"] == dti) & (black_denied["age_group"] == age)]
    w = white_approved[(white_approved["debt_to_income_ratio"] == dti) & (white_approved["age_group"] == age)]
    distance = cdist(((b[numbers] - mean) / scale).to_numpy(), ((w[numbers] - mean) / scale).to_numpy())
    row, column = np.unravel_index(np.argmin(distance), distance.shape)
    if best is None or distance[row, column] < best[0]:
        best = (distance[row, column], b.iloc[row], w.iloc[column])

_, denied, match = best


def describe(row):
    return {
        "race": row["race"],
        "age": row["age_group"],
        "loan amount": f"${row['loan_amount']:,.0f}",
        "loan-to-value": f"{row['loan_to_value_ratio']:.0f}",
        "income ($000s)": f"{row['income']:,.0f}",
        "debt-to-income": str(row["debt_to_income_ratio"]),
        "term (months)": f"{row['loan_term']:,.0f}",
        "model score": f"{row['score']:.3f}",
        "recorded outcome": "approved" if row["target"] == 1 else "denied",
    }


pd.DataFrame({
    "Black applicant (model denies)": describe(denied),
    "White applicant (model approves)": describe(match),
})

,Black applicant (model denies),White applicant (model approves)
race,Black,White
age,25 and over,25 and over
loan amount,"$305,000","$305,000"
loan-to-value,96,96
income ($000s),69,69
debt-to-income,50%-60%,50%-60%
term (months),360,360
model score,0.716,0.874
recorded outcome,denied,approved


**What this shows.** The closest Black-denied / White-approved pair in the test set: same debt-to-income bucket, same age group, and loan size, loan-to-value and income within a few percent of each other. The model sends them opposite ways. That is not a bug in the code; it is the model learning the pattern in the recorded decisions, race included.

One pair can be a coincidence, which is why the headline number is the gap across 20,000 held-out applications. The pair is here because a counselor sees one application at a time, and this is what that looks like.

## 7. Our addition A: remove every demographic column

The obvious fix is to delete race, ethnicity, sex and age, and retrain. If the gap came from the columns, it should disappear.

In [8]:
blind = make_model(blind_features).fit(X.iloc[train], y[train])
p_blind_val = blind.predict_proba(X.iloc[val])[:, 1]
p_blind_test = blind.predict_proba(X.iloc[test])[:, 1]
cut_blind = calibrated_cut(y[val], p_blind_val)
approved_blind = p_blind_test >= cut_blind

for trait in ("race", "age"):
    report(f"Blind model, {trait}", approved_blind, y[test], trait,
           df[dataset.GROUPS[trait]["column"]].to_numpy()[test])

Blind model, race
  accuracy         87.7%
  Black            75.9%    White          86.6%    gap -10.7
Blind model, age
  accuracy         87.7%
  under 25         84.9%    25 and over    84.7%    gap +0.2


It barely moves: **−10.7** on this split instead of −18.7 (50 splits: −11.2 against −19.7). Blinding is not a fix, because the remaining columns still describe the same applicants. To show that, we train a second model whose only job is to guess the applicant's race from the columns the blind model is allowed to see. If that model beats a coin flip, the trait is still in the data.

In [9]:
from sklearn.metrics import roc_auc_score

is_black = (df["race"] == "Black").to_numpy()
probe = make_model(blind_features).fit(X.iloc[train], is_black[train])
auc = roc_auc_score(is_black[test], probe.predict_proba(X.iloc[test])[:, 1])
print(f"proxy AUC: {auc:.2f}   (0.50 = the remaining columns say nothing about race)")

proxy AUC: 0.84   (0.50 = the remaining columns say nothing about race)


In [10]:
PRETTY = {
    "loan_to_value_ratio": "loan-to-value ratio",
    "debt_to_income_ratio": "debt-to-income ratio",
    "loan_amount": "loan amount",
    "property_value": "property value",
    "tract_minority_population_percent": "tract minority share",
    "tract_to_msa_income_percentage": "tract income vs metro",
    "ffiec_msa_md_median_family_income": "metro median family income",
    "derived_loan_product_type": "loan product",
    "derived_dwelling_category": "dwelling type",
    "aus-1": "underwriting system",
    "loan_type": "loan type",
    "lien_status": "lien",
    "construction_method": "construction",
    "occupancy_type": "occupancy",
    "preapproval": "preapproval",
    "initially_payable_to_institution": "payable to this lender",
    "reverse_mortgage": "reverse mortgage",
    "open-end_line_of_credit": "credit line",
    "business_or_commercial_purpose": "purpose",
    "submission_of_application": "how it was submitted",
    "total_units": "units",
    "conforming_loan_limit": "conforming limit",
}

# HMDA stores most of these as numeric codes; translate the ones a person reads.
CODES = {
    "loan_type": {"1": "Conventional", "2": "FHA", "3": "VA", "4": "USDA/RHS"},
    "lien_status": {"1": "first lien", "2": "subordinate lien"},
    "construction_method": {"1": "site-built", "2": "manufactured"},
    "occupancy_type": {"1": "principal residence", "2": "second home", "3": "investment property"},
    "preapproval": {"1": "preapproval requested", "2": "no preapproval"},
    "initially_payable_to_institution": {"1": "yes", "3": "no"},
    "reverse_mortgage": {"1": "yes", "2": "no"},
    "open-end_line_of_credit": {"1": "open-end", "2": "closed-end"},
    "business_or_commercial_purpose": {"1": "business", "2": "personal"},
    "submission_of_application": {"1": "direct", "2": "not direct", "3": "not applicable", "7": "other"},
    "aus-1": {"1": "DU", "2": "LPA", "3": "TOTAL Scorecard", "4": "GUS",
              "5": "other system", "6": "no system", "7": "lender's own system"},
    "total_units": {"1": "1 unit", "2": "2 units", "3": "3 units", "4": "4 units", "5": "5-24 units"},
    "conforming_loan_limit": {"C": "conforming", "NC": "non-conforming", "U": "unknown"},
}


def feature_labels(model):
    """Turn pipeline feature names into something a person can read."""
    raw = model.named_steps["transform"].get_feature_names_out()
    labels = []
    for name in raw:
        if name.startswith("num__"):
            body = name[5:]
            labels.append(PRETTY.get(body, body.replace("_", " ")))
            continue
        body = name[5:]
        for column in sorted(dataset.CATEGORICAL, key=len, reverse=True):
            if body.startswith(column + "_"):
                value = body[len(column) + 1:]
                value = CODES.get(column, {}).get(value, value)
                labels.append(f"{PRETTY.get(column, column.replace('_', ' '))} = {value}")
                break
        else:
            labels.append(body)
    return np.array(labels)


coefficients = probe.named_steps["model"].coef_[0]
labels = feature_labels(probe)
strongest = np.argsort(coefficients)[-8:][::-1]
pd.DataFrame({"column": labels[strongest], "weight": coefficients[strongest].round(3)})

,column,weight
0,tract minority share,1.120
1,underwriting system = lender's own system,0.565
2,debt-to-income ratio = >60%,0.540
3,debt-to-income ratio = 50%-60%,0.352
4,units = 3 units,0.317
5,loan amount,0.264
6,underwriting system = no system,0.253
7,loan product = FSA/RHS:Subordinate Lien,0.246


The proxy model reaches **AUC 0.84** — far from chance — and the columns doing the work are the census tract's minority share, the underwriting system, debt-to-income and loan size. Dropping the race column does not remove race from the data; it removes the *evidence* that race was used. That distinction is the whole argument of a disparate-impact audit.

## 8. Our addition B: a separate cutoff per group

If the blind model scores the two groups differently, one way to equalise approvals is to set the cutoff per group so that each group is approved at the same rate. This is the strategy the page calls **blind + group cutoffs**. It is the strongest fix in the audit, and the least usable.

In [11]:
from audit.audit import group_cutoffs

g_val = df["race"].to_numpy()[val]
g_test = df["race"].to_numpy()[test]
overall_rate = float((p_blind_val >= cut_blind).mean())      # the rate the blind model approves at
cuts = group_cutoffs(p_blind_val, g_val, overall_rate)
approved_cut = p_blind_test >= np.array([cuts[g] for g in g_test])

print("cutoffs:", {g: round(c, 3) for g, c in cuts.items()})
report("Blind model + group cutoffs, race", approved_cut, y[test], "race", g_test)
print(f"accuracy: {(approved_blind == y[test]).mean() * 100:.1f}% → {(approved_cut == y[test]).mean() * 100:.1f}%")

cutoffs: {'Black': 0.554, 'White': 0.792}
Blind model + group cutoffs, race
  accuracy         87.1%
  Black            85.0%    White          84.2%    gap +0.9
accuracy: 87.7% → 87.1%


The gap falls to about zero (**+0.9** on this split, **+0.1** across 50) for roughly half a point of accuracy. The catch is not statistical. To apply a different cutoff you must read the applicant's race at decision time, which is precisely the disparate treatment that survived the 2026 amendment to Regulation B. **Parity and legality are different tests**, and a gap of zero does not make a strategy shippable.

The other rules — equal true-positive rate, equal false-positive rate, equal error — are in `results/cutoffs.csv`. On the blind model, matching approval rates closes the gap; matching error rates only gets it to −3.1.

## 9. The demo: the tool explains a decision

This is the part a counselor would actually use. They paste one application in, and the tool says whether this model would approve it and which columns pushed the decision, so the applicant learns what to fix — or sees that the numbers were fine and the group was not.

The reasons come from the model itself: each column's value multiplied by its learned weight, sorted by how much it moved the score.

In [12]:
def explain(index, model=blind, cutoff=cut_blind, top=3):
    row = df.loc[index]
    frame = pd.DataFrame([row[blind_features]])
    transform = model.named_steps["transform"]
    weights = model.named_steps["model"].coef_[0]
    contribution = np.asarray(transform.transform(frame).todense()).ravel() * weights
    labels = feature_labels(model)
    score = float(model.predict_proba(frame)[0, 1])

    # Leave out the columns that barely moved the score, so the reasons stay readable.
    floor = 0.15 * max(abs(contribution).max(), 1e-9)
    for_it = [labels[i] for i in np.argsort(contribution)[::-1]
              if contribution[i] > floor][:top]
    against = [labels[i] for i in np.argsort(contribution)
               if contribution[i] < -floor][:top]
    verdict = "⚠️  Looks like a denial" if score < cutoff else "✅  Looks like an approval"
    print(f"{verdict} — score {score:.4f} against a cutoff of {cutoff:.4f} ({100 * (score - cutoff):+.2f} points)")
    print(f"   counts for the applicant: {', '.join(for_it) if for_it else 'not much'}")
    print(f"   counts against: {', '.join(against) if against else 'nothing'}")
    print()


candidates = df.iloc[test].copy()
candidates["blind score"] = p_blind_test
candidates = candidates.dropna(subset=["loan_amount", "loan_to_value_ratio", "income", "property_value"])

near = candidates[(candidates["blind score"] - cut_blind).abs() < 0.2]
close_denial = near[near["blind score"] < cut_blind].sort_values("blind score").iloc[-1]
close_approval = near[near["blind score"] >= cut_blind].sort_values("blind score").iloc[0]
disagreement = candidates[(candidates["blind score"] >= cut_blind + 0.15) & (candidates["target"] == 0)]
disagreement = disagreement.sort_values("blind score").iloc[0]

for index, why in [
    (close_denial.name, "just below the line"),
    (disagreement.name, "the model would approve it, the lender did not"),
    (close_approval.name, "just above the line"),
]:
    outcome = "approved" if y[index] == 1 else "denied"
    print(f"application {index} — recorded outcome: {outcome}, {why}")
    explain(index)

application 10168 — recorded outcome: approved, just below the line
⚠️  Looks like a denial — score 0.7595 against a cutoff of 0.7597 (-0.02 points)
   counts for the applicant: payable to this lender = yes, construction = site-built
   counts against: preapproval = no preapproval, underwriting system = no system, reverse mortgage = no

application 91129 — recorded outcome: denied, the model would approve it, the lender did not
✅  Looks like an approval — score 0.9097 against a cutoff of 0.7597 (+15.01 points)
   counts for the applicant: payable to this lender = yes, construction = site-built, debt-to-income ratio = 44
   counts against: preapproval = no preapproval, underwriting system = no system, reverse mortgage = no

application 78302 — recorded outcome: approved, just above the line
✅  Looks like an approval — score 0.7598 against a cutoff of 0.7597 (+0.01 points)
   counts for the applicant: payable to this lender = yes, underwriting system = TOTAL Scorecard, construction = sit

Each line is a column the applicant can see on their own form, which is the point: the explanation has to be checkable by the person it is about. When the reasons are all financial, the counselor can work on them. When the model would deny an application whose reasons look ordinary, that is the case worth escalating.

Some of the lines are programme flags from the HMDA form rather than financial facts — preapproval, reverse mortgage, payable to this lender. They are real model inputs, and the audit keeps them, but the columns a counselor can act on are the ratios, the loan size and the income.

The web page in `results/index.html` shows the same audit for both traits, with the strategies as buttons and the confidence bands drawn as bars.

## 10. Results summary

The page and the write-up report 50 random 60/20/20 splits. Re-running all 50 here would take a couple of minutes, so this cell runs **20** and prints the same table; the numbers move by a few tenths of a point, and `results/audit.json` holds the 50-split version.

In [13]:
from audit.audit import run

res = run(splits=20, seed=0)
rows = []
for trait, strategies in res["res"].items():
    for name, v in strategies.items():
        rows.append({
            "trait": trait,
            "strategy": name,
            "other": v[0],
            "ref": v[1],
            "accuracy": v[2],
            "gap": round(v[0] - v[1], 1),
            "95% range across splits": f"{v[3]:+.1f} to {v[4]:+.1f}",
        })
pd.DataFrame(rows)

,trait,strategy,other,ref,accuracy,gap,95% range across splits
0,age,approve everyone,100.0,100.0,84.5,0.0,+0.0 to +0.0
1,age,observed in data,85.2,84.6,NaN,0.6,-0.8 to +2.8
2,age,full model,85.1,84.5,88.0,0.6,-1.0 to +2.3
3,age,blind model,84.2,84.6,87.8,-0.4,-2.2 to +1.4
4,age,blind + group cutoffs,84.1,84.6,87.8,-0.5,-3.7 to +2.5
5,race,approve everyone,100.0,100.0,84.5,0.0,+0.0 to +0.0
6,race,observed in data,74.8,86.8,NaN,-12.0,-12.8 to -11.0
7,race,full model,68.6,88.0,88.0,-19.4,-21.4 to -17.5
8,race,blind model,75.5,86.5,87.8,-11.0,-12.1 to -9.6
9,race,blind + group cutoffs,84.9,84.5,87.4,0.4,-0.8 to +1.6


In [14]:
print(f"proxy AUC — race {res['proxy']['race']:.2f}, age {res['proxy']['age']:.2f}")
print(f"base rate {res['base']}% · {res['rows']:,} applications · {res['splits']} splits")

proxy AUC — race 0.84, age 0.76
base rate 84.5% · 100,000 applications · 20 splits


## 11. Limitations and next steps

- **This is not a lender's model.** HMDA has no credit score, no assets and no employment history, and the demographics are self-reported with large "not available" shares. What we measure is the pattern in public data, not a specific bank's algorithm.
- **Four states, two loan purposes, one year.** An earlier run on Oregon purchase-only applications showed a −3.2 age gap that vanished when we widened the sample; conclusions from one slice do not travel.
- **An approval is not a fair loan.** Pricing, fees and terms decide whether a borrower is treated fairly, and none of them are in this audit — we exclude them because they are recorded after the decision.
- **Group cutoffs need the protected trait.** Our best-performing fix is also the one that cannot legally ship. The genuinely useful next step is a fix that works without reading race: reweighting the training data, dropping the census-tract columns that act as proxies, or auditing a lender's real scorecard.
- **Next, with more time:** test whether the gap holds in 2023–2024 as well as 2025, and measure what happens to both the proxy AUC and accuracy when the tract columns are removed.

## Sources

- HMDA loan-level data, CFPB Data Browser API (activity year 2025, states GA, MD, MI, AZ, loan purposes 1 and 31, actions taken 1 and 3): <https://ffiec.cfpb.gov/data-publication/>
- Regulation B final rule, 91 FR 21620 (Federal Register document 2026-07804), effective 21 July 2026: <https://www.federalregister.gov/d/2026-07804>
- *National Fair Housing Alliance v. CFPB*, No. 1:26-cv-01820 (D.D.C.) — the pending challenge to that rule.
- scikit-learn (ColumnTransformer, LogisticRegression, ROC AUC), pandas, NumPy.
- The page design and the `audit/` pipeline are ours; `results/index.html` is generated from `results/audit.json`.
- **AI tools:** we used an AI coding assistant (Codebuff) to help write and debug the pipeline and this notebook. Every number here is produced by the code in this repository when it runs, and the recorded approval rates were checked against the raw HMDA file.

*This project is a prototype and is not financial advice.*